# 산불감지 OOD 재설계 학습 — Colab L4

**그냥 [런타임 → 모두 실행] 누르세요.** (Drive 인증 팝업 1번만 클릭)

- 데이터: 영상 30편 다양화 (정상 792 / 화재 727)
- 설정 E (대칭 강증강), 7모델 × 3fold, epoch 15
- L4 기준 ~2시간. 끊겨도 학습 셀만 다시 실행하면 이어서 진행.

**준비물**: Google Drive(MyDrive)에 `fireimage_clean.zip` 업로드 (PC Drive 계정 = Colab 계정 동일해야 함)

In [ ]:
# 1. GPU 확인 + 코드 클론
import torch, shutil, os
assert torch.cuda.is_available(), 'GPU 런타임 필요 (런타임→유형변경→L4)'
print('GPU:', torch.cuda.get_device_name(0))
if os.path.exists('/content/fireimage_detection'):
    shutil.rmtree('/content/fireimage_detection')
!git clone https://github.com/yuntaewon812/fireimage_detection.git /content/fireimage_detection
%cd /content/fireimage_detection

In [ ]:
# 2. 패키지 설치
!pip install timm einops transformers -q
print('패키지 설치 완료')

In [ ]:
# 3. Drive 연결 + 데이터 압축해제 + 결과 영구저장 설정
from google.colab import drive
import os, glob, zipfile, shutil
drive.mount('/content/drive')

# 가중치/결과를 Drive에 영구저장 (끊겨도 이어학습)
CKPT = '/content/drive/MyDrive/fireimage_ablation'
for sub in ['model_save', 'results']:
    os.makedirs(f'{CKPT}/{sub}', exist_ok=True)
    link = f'/content/fireimage_detection/{sub}'
    if os.path.islink(link): os.unlink(link)
    elif os.path.exists(link): shutil.rmtree(link, ignore_errors=True)
    os.symlink(f'{CKPT}/{sub}', link)

# 데이터 압축해제 (Drive의 zip)
BASE = '/content/fireimage_detection/data/fireimage'
if os.path.exists(BASE): shutil.rmtree(BASE)
os.makedirs(BASE, exist_ok=True)
ZIP = '/content/drive/MyDrive/fireimage_clean.zip'
assert os.path.exists(ZIP), f'Drive에 zip 없음: {ZIP}
PC Drive 계정과 Colab 로그인 계정이 같은지 확인하세요'
with zipfile.ZipFile(ZIP) as z:
    z.extractall(BASE)

IMG = ('.jpg','.jpeg','.png','.bmp')
n = len(glob.glob(f'{BASE}/normal/**/*', recursive=True))
a = len(glob.glob(f'{BASE}/abnormal/**/*', recursive=True))
print(f'데이터 준비 완료 — normal {n} / abnormal {a}')
assert n > 0 and a > 0

In [ ]:
# 4. 학습 (7모델 × 3fold, 설정 E, epoch 15) — L4로 ~2시간
%cd /content/fireimage_detection
!python main_ablation.py --class_name fireimage --setting E --epochs 15 --patience 5

In [ ]:
# 5. 결과 확인 (OOD 포함 fold별 F1)
import pandas as pd, os
p = 'results/fireimage_abl_E/metrics.csv'
if os.path.exists(p):
    df = pd.read_csv(p)
    print(df[['model name','F1 score','AUROC']].to_string())
else:
    print('아직 결과 없음 — 학습 셀(4)이 끝나야 생성됨')